In [1]:
# Bagging
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

import numpy as np
import pandas as pd

In [2]:
# load the dataset, built-in
data = load_breast_cancer()
X, y = data.data, data.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state = 10202025)

data

{'data': array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
         1.189e-01],
        [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
         8.902e-02],
        [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
         8.758e-02],
        ...,
        [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
         7.820e-02],
        [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
         1.240e-01],
        [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
         7.039e-02]]),
 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
        1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
        1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0

In [41]:
# simple DT as the baseline
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(random_state = 10202025)

In [42]:
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test) 
y_prob_tree = tree.predict_proba(X_test)[:,1] # class 1's probability

In [43]:
from sklearn.metrics import accuracy_score , roc_auc_score
acc_tree = accuracy_score(y_test, y_pred_tree)
auc_tree = roc_auc_score(y_test, y_prob_tree)

In [44]:
print(f"Accuracy: {acc_tree}")
print(f"ROC AUC:  {auc_tree}")

Accuracy: 0.916083916083916
ROC AUC:  0.9151272577996715


In [57]:
# Bagging
from sklearn.ensemble import BaggingClassifier
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(random_state = 10202025),
    n_estimators=100,
    max_samples = 1,
    bootstrap = True
)
bag.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(random_state=10202025),
                  max_samples=1, n_estimators=100)

In [58]:
y_pred_bag = bag.predict(X_test) 
y_prob_bag = bag.predict_proba(X_test)[:,1]
acc_bag = accuracy_score(y_test, y_pred_bag)
auc_bag = roc_auc_score(y_test, y_prob_bag)

In [59]:
print(f"Bag Accuracy: {acc_bag}")
print(f"Bag ROC AUC:  {auc_bag}")

Bag Accuracy: 0.6083916083916084
Bag ROC AUC:  0.5


In [63]:
# Random Forst = Bagging + Random feature subsampling
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    random_state = 10202025,
    n_estimators=100,
    max_depth = None
)
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=10202025)

In [64]:
y_pred_rf = rf.predict(X_test) 
y_prob_rf = rf.predict_proba(X_test)[:,1]
acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)
print(f"Bag Accuracy: {acc_rf}")
print(f"Bag ROC AUC:  {auc_rf}")

Bag Accuracy: 0.9370629370629371
Bag ROC AUC:  0.9834770114942528


In [69]:
# cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state = 10202025)
score_tree = cross_val_score(tree, X, y, cv=cv, scoring="accuracy")
score_bag = cross_val_score(bag, X, y, cv=cv, scoring="accuracy")
score_rf = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")
# print(score_tree.std())
print(f"Decision Tree: {score_tree.mean()} +- {score_tree.std()}")
print(score_bag.mean())
print(score_rf.mean())

Decision Tree: 0.9174041297935103 +- 0.020437413555236823
0.6274181027790716
0.9577860580655176


In [71]:
# Boosting
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

ada = AdaBoostClassifier(
    estimator = DecisionTreeClassifier(max_depth=1, random_state = 10202025),
    n_estimators= 200,
    learning_rate=0.1,
)

ada.fit(X_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,
                                                    random_state=10202025),
                   learning_rate=0.1, n_estimators=200)

In [72]:
y_pred_ada = ada.predict(X_test) 
y_prob_ada = ada.predict_proba(X_test)[:,1]
acc_ada = accuracy_score(y_test, y_pred_ada)
auc_ada = roc_auc_score(y_test, y_prob_ada)
print(f"ada Accuracy: {acc_ada}")
print(f"ada ROC AUC:  {auc_ada}")

ada Accuracy: 0.9370629370629371
ada ROC AUC:  0.9694170771756978


In [74]:
#Stacking
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [77]:
base_learners = [
    ("rf", RandomForestClassifier(random_state = 10202025, n_estimators=100)), 
    ("gb", GradientBoostingClassifier(random_state = 10202025)),
    ("svc",make_pipeline(StandardScaler(),SVC(probability=True, random_state = 10202025))),
    ("knn", make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=15))),
]

In [79]:
meta_learner = LogisticRegression(max_iter=500, random_state = 10202025)

In [82]:
from sklearn.ensemble import StackingClassifier
stack = StackingClassifier(
    estimators= base_learners,
    final_estimator=meta_learner,
    stack_method= "predict_proba",
    passthrough=False,
    cv=5
)
stack.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('rf',
                                RandomForestClassifier(random_state=10202025)),
                               ('gb',
                                GradientBoostingClassifier(random_state=10202025)),
                               ('svc',
                                Pipeline(steps=[('standardscaler',
                                                 StandardScaler()),
                                                ('svc',
                                                 SVC(probability=True,
                                                     random_state=10202025))])),
                               ('knn',
                                Pipeline(steps=[('standardscaler',
                                                 StandardScaler()),
                                                ('kneighborsclassifier',
                                                 KNeighborsClassifier(n_neighbors=15))]))],
                   final_estimator=LogisticRegression(max_iter=500,
                                                      random_state=10202025),
                   stack_method='predict_proba')

y_pred_stack = stack.predict(X_test) 
y_prob_stack = ada.predict_proba(X_test)[:,1]
acc_stack = accuracy_score(y_test, y_pred_stack)
auc_stack = roc_auc_score(y_test, y_prob_stack)
print(f"stack Accuracy: {acc_stack}")
print(f"stack ROC AUC:  {auc_stack}")

In [85]:
y_pred_stack = stack.predict(X_test) 
y_prob_stack = stack.predict_proba(X_test)[:,1] 
acc_stack = accuracy_score(y_test, y_pred_stack) 
auc_stack = roc_auc_score(y_test, y_prob_stack) 
print(f"stack Accuracy: {acc_stack}")
print(f"stack ROC AUC: {auc_stack}")

stack Accuracy: 0.958041958041958
stack ROC AUC: 0.986247947454844
